# PAT-1101

In [ ]:
A, D = input().split()
D = int(D)
B = A[-D:] + A[:-D]
#result = round((int(B)/int(A)), 2)
result = int(B)/int(A)
print(f"{result:.2f}")

# PAT-1102

In [ ]:
N = int(input())
#初始化字典，存储商品信息：{商品名: [单价, 销售量]}
data_input = {}
for i in range(N):
    A, B, C = input().split()
    B = int(B)
    C = int(C)
    data_input[A] = [B, C]

# 最大销售量
max_num = max(data_input, key=lambda x: data_input[x][1])
print(max_num, data_input[max_num][1])

# 最大销售额
max_money = max(data_input, key=lambda x: data_input[x][0]*data_input[x][1])
print(max_money, data_input[max_money][1]*data_input[max_money][0])


# PAT-1103
1. $a^3-(a-1)^3=c^2$
2. $b^2 + (b-1)^2=c$
3. 题目给定a的范围[1,25000]，只要遍历a，算出c，通过c，去找b就行。
4. 因为c是两整数的平方和，所以只要对c为整数的情况进行判断就行。即```int(c)==c```.
5. b与c的函数又可简化成$2x^2=c$，由此可推断出b就在```int(x)```的左边或者右边。
6. 注意到```x>b```，所以```b=int(x)```或者```b=int(x)+1```
6. 此时只要验证```int(x)-1```或者```int(x)+1```就行
7. **b要从2开始取值**

In [ ]:
m, n = map(int, input().split())
found = False
for i in range(m, n+1):
    may_c = (i**3 - (i-1)**3)**0.5
    c = int(may_c)

    # 判断c是否为整数
    if c == may_c:
        # 找出b的大概范围
        may_b = int((c/2)**0.5)
        # 对b左右两边的两个数进行精确验证
        # 限定b至少为2
        if may_b >= 2:
            if (may_b)**2 + (may_b-1)**2 == c:
                print(i, may_b)
                found = True
                continue
            if (may_b)**2 + (may_b+1)**2 == c:
                print(i, may_b+1)
                found = True
                continue
if not found:
    print("No Solution")

# PAT-1104 天长地久
1. 打表发现，符合条件的数，都已99结尾


好的，我来详细解释这个优化的数学原理和实现细节。

## 核心数学原理：6k ± 1 定理

### 1. 数学基础
所有大于3的整数都可以表示为以下六种形式之一：
- 6k
- 6k + 1  
- 6k + 2
- 6k + 3
- 6k + 4
- 6k + 5 (等价于 6k - 1)

### 2. 排除非质数形式
- **6k**：能被6整除 → 非质数
- **6k + 2**：能被2整除 → 非质数  
- **6k + 3**：能被3整除 → 非质数
- **6k + 4**：能被2整除 → 非质数

### 3. 只剩下两种可能形式
- **6k + 1**
- **6k + 5** (即 6k - 1)

**结论**：所有质数（除了2和3）都必须是 6k ± 1 的形式！

## 代码实现详解

```python
def is_prime(num):
    # 处理边界情况
    if num <= 1:
        return False    # 1及以下的数不是质数
    if num <= 3:
        return True     # 2和3是质数
    
    # 排除能被2或3整除的数
    if num % 2 == 0 or num % 3 == 0:
        return False
    
    # 核心优化：只检查 6k ± 1 形式的因子
    i = 5               # 从5开始（第一个6k-1形式的数）
    while i * i <= num: # 只需要检查到√num
        # 检查 i (6k-1) 和 i+2 (6k+1) 是否能整除num
        if num % i == 0 or num % (i + 2) == 0:
            return False
        i += 6          # 每次增加6，跳到下一个6k-1位置
    
    return True
```

## 具体执行过程示例

以检查 **97** 是否为质数为例：

### 初始检查：
- 97 > 1 ✓
- 97 > 3 ✓  
- 97 % 2 ≠ 0 ✓
- 97 % 3 ≠ 0 ✓

### 循环检查：
```
i = 5: 检查 5 和 7
  97 % 5 = 2 ≠ 0
  97 % 7 = 6 ≠ 0

i = 11: 检查 11 和 13  
  97 % 11 = 9 ≠ 0
  97 % 13 = 6 ≠ 0

i = 17: 检查 17 和 19
  97 % 17 = 12 ≠ 0
  97 % 19 = 2 ≠ 0

i = 23: 23² = 529 > 97，停止循环
```

## 为什么这样优化？

### 传统方法的问题：
```python
# 传统方法：检查所有2到√n的数
for i in range(2, int(num**0.5) + 1):
    if num % i == 0:
        return False
```

这会检查很多明显不可能整除的数（偶数、3的倍数等）

### 优化后的优势：
- **检查次数减少约2/3**
- **跳过所有偶数和3的倍数**
- **只检查真正有可能的候选因子**

## 时间复杂度分析

虽然时间复杂度仍然是 **O(√n)**，但：

- **传统方法**：检查约 √n 个数
- **优化方法**：检查约 √n / 3 个数

**实际性能提升**：大约2-3倍的速度提升！

## 数学证明

为什么只需要检查 6k ± 1 形式的因子？

假设 `n` 有一个因子 `d`，那么：
1. 如果 `d` 能被2或3整除，那么 `n` 也能被2或3整除
2. 我们已经提前检查了2和3的整除性
3. 因此，任何其他因子都必须是 6k ± 1 的形式

这个优化基于坚实的数论基础，是判断质数的最优单线程算法之一。

In [ ]:
import math

N = int(input())

# 判断素数的函数，在本题中，限定了num必须大于2才算素数
def is_prime(num):
    if num <= 2:
        return False
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True

# 更高效的素数判断函数
def is_prime_2(num):
    if num <= 1:
        return False
    if num <= 3:
        return True
    if num % 2 == 0 or num % 3 == 0:
        return False
    
    # 检查 6k ± 1 形式的数
    i = 5
    while i * i <= num:
        if num % i == 0 or num % (i + 2) == 0:
            return False
        i += 6
    
    return True

def generate_numbers(d, s, prefix='', results=None):
    if results is None:
        results = []
    # 递归结束：长度达到 d 且和为 0
    if d == 0 and s == 0:
        results.append(int(prefix))
        return results
    if d == 0 or s < 0:
        return results
    
    # 确定当前位的范围
    start = 0 if prefix else 1  # 第一位不能是 0（除非 d=1）
    for digit in range(start, 10):
        generate_numbers(d-1, s-digit, prefix+str(digit), results)
    
    return results



for i in range(N):
    k, m = map(int, input().split())
    numbers = generate_numbers(k, m)
    if not numbers:
        print(f'Case {i+1}\nNo Solution')
        continue
    else:
        found = False
        for number in sorted(numbers):
            if number % 100 == 99:
                n = sum(list(map(int, str(number+1))))
                max_gcd = math.gcd(m, n)
                if is_prime(max_gcd):
                    if not found:
                        found = True
                        print(f'Case {i+1}')
                    print(f'{n} {number}')



Case 1
10 189999
10 279999
10 369999
10 459999
10 549999
10 639999
10 729999
10 819999
10 909999
Case 2
No Solution


In [ ]:
import math
 
 

def is_prime(num): 
    if num < 2: 
        return False
    for x in range(2, int(num ** 0.5) + 1):
        if num % x == 0:
            return False
    return True
 
 
def found_nums(lst, length, s, k, m):  
    """
    使用深度搜索，结合回溯算法
    lst: 当前数字列表
    length: 当前数字长度
    s: 当前数字和
    k: 目标数字长度
    m: 目标数字和
    """
    # 提前剪枝
    if s + (k - length) * 9 < m:  
        return
    
    # 迭代出口
    if length == k:  
        if s == m:  
            res = ''.join(lst)  
            n = sum(int(d) for d in str(int(res) + 1)) 
            p = math.gcd(n, m) 
            if p > 2 and is_prime(p):  
                ans.append(f"{n} {res}")  
        return
    
    # 核心迭代递归代码
    for t in range(10): 
        if s + t <= m: # 剪枝
            tmp = lst[length] # 记录当前位的值，便于回溯
            lst[length] = str(t) # 设置当前位
            found_nums(lst, length + 1, s + t, k, m) # 递归调用，深入下一位
            lst[length] = tmp # 回溯
    return
 
 

# 获取首行输入数据4
N = int(input())
 
# 计算结果并输出
for i in range(N):
    print(f"Case {i+1}")
    K, M = map(int, input().split())
    ans = []
    a = ['0'] * K
    for j in range(1, 10):
        a[0] = str(j)
        found_nums(a, 1, j, K, M)

    ans.sort(key=lambda x: int(x.split()[0]))

    print('\n'.join(ans) if ans else 'No Solution')


Case 1
Debug: ans before sort: ['10 189999', '10 279999', '10 369999', '10 459999', '10 549999', '10 639999', '10 729999', '10 819999', '10 909999']
Debug: ans length: 9
<class 'list'>
10 189999
10 279999
Debug: ans after sort: ['10 189999', '10 279999', '10 369999', '10 459999', '10 549999', '10 639999', '10 729999', '10 819999', '10 909999']
10 189999
10 279999
10 369999
10 459999
10 549999
10 639999
10 729999
10 819999
10 909999


### ASCII 搜索树（基于上方 found_nums 规则）
下面工具用于为给定的 K, M 构建完整搜索树（首位 1-9，其余 0-9，含原剪枝：s + (k - length) * 9 < m），并以 ASCII 形式输出：
- 叶子 depth==K 一定满足位和 = M（否则在上一层被剪掉）
- 若叶子同时满足 gcd(sum_digits(number+1), M) 为素数且 > 2，则标记 [OK]
- 可能节点数巨大，请只在较小 K(≤4) 时打印；可通过 max_leaves/ max_nodes 限制规模。

In [7]:
import math
from typing import List, Optional

class Node:
    __slots__ = ("digit","s","depth","children","is_leaf","ok")
    def __init__(self,digit:Optional[int],s:int,depth:int):
        self.digit = digit      # 根节点为 None
        self.s = s              # 累计和
        self.depth = depth      # 已放置位数
        self.children: List['Node'] = []
        self.is_leaf = False
        self.ok = False         # 是否满足 gcd(sum_digits(num+1), M) 是素数且 >2

    def label(self):
        base = "root" if self.digit is None else str(self.digit)
        tag = " [OK]" if self.ok else ""
        return f"{base}(s={self.s},d={self.depth}){tag}"


def build_tree(K:int,M:int,max_nodes:int=50000)->Node:
    global_nodes = 0
    root = Node(None,0,0)

    def sum_digits(x:int)->int:
        return sum(int(c) for c in str(x))

    def expand(node:Node,prefix_digits:List[int]):
        nonlocal global_nodes
        # 剪枝：最大理论可加和不足
        if node.s + (K - node.depth)*9 < M:
            return
        if node.depth == K:
            if node.s == M:
                node.is_leaf = True
                # 计算 number+1 的数字和
                number = 0
                for d in prefix_digits:
                    number = number*10 + d
                n = sum_digits(number+1)
                g = math.gcd(n,M)
                if g > 2 and is_prime(g):
                    node.ok = True
            return
        start = 1 if node.depth==0 else 0
        for t in range(start,10):
            if node.s + t <= M:
                child = Node(t,node.s + t,node.depth+1)
                node.children.append(child)
                global_nodes += 1
                if global_nodes >= max_nodes:
                    return
                prefix_digits.append(t)
                expand(child,prefix_digits)
                prefix_digits.pop()

    expand(root,[])
    return root


def render_ascii(root:Node,max_leaves:int=5000):
    lines: List[str] = []
    leaves = 0
    def dfs(node:Node,prefix:str,is_last:bool):
        nonlocal leaves
        connector = "└─ " if is_last else "├─ "
        if node.digit is None:
            lines.append(node.label())
        else:
            lines.append(prefix + connector + node.label())
        if node.is_leaf:
            leaves += 1
            if leaves > max_leaves:
                lines.append(f"... 叶子超过上限 {max_leaves}，停止渲染 ...")
                return
        if node.children and leaves <= max_leaves:
            new_prefix = prefix + ("   " if is_last else "│  ")
            for i,ch in enumerate(node.children):
                if leaves > max_leaves:
                    break
                dfs(ch,new_prefix,i==len(node.children)-1)
    dfs(root,"",True)
    return lines


def print_search_tree(K:int,M:int,max_nodes:int=50000,max_leaves:int=5000):
    root = build_tree(K,M,max_nodes=max_nodes)
    for line in render_ascii(root,max_leaves=max_leaves):
        print(line)


In [8]:
# 示例：打印 K=3, M=6 的搜索树（可自行修改参数）
print_search_tree(K=3, M=6, max_nodes=20000, max_leaves=3000)

root(s=0,d=0)
   ├─ 1(s=1,d=1)
   │  ├─ 0(s=1,d=2)
   │  │  ├─ 0(s=1,d=3)
   │  │  ├─ 1(s=2,d=3)
   │  │  ├─ 2(s=3,d=3)
   │  │  ├─ 3(s=4,d=3)
   │  │  ├─ 4(s=5,d=3)
   │  │  └─ 5(s=6,d=3)
   │  ├─ 1(s=2,d=2)
   │  │  ├─ 0(s=2,d=3)
   │  │  ├─ 1(s=3,d=3)
   │  │  ├─ 2(s=4,d=3)
   │  │  ├─ 3(s=5,d=3)
   │  │  └─ 4(s=6,d=3)
   │  ├─ 2(s=3,d=2)
   │  │  ├─ 0(s=3,d=3)
   │  │  ├─ 1(s=4,d=3)
   │  │  ├─ 2(s=5,d=3)
   │  │  └─ 3(s=6,d=3)
   │  ├─ 3(s=4,d=2)
   │  │  ├─ 0(s=4,d=3)
   │  │  ├─ 1(s=5,d=3)
   │  │  └─ 2(s=6,d=3)
   │  ├─ 4(s=5,d=2)
   │  │  ├─ 0(s=5,d=3)
   │  │  └─ 1(s=6,d=3)
   │  └─ 5(s=6,d=2)
   │     └─ 0(s=6,d=3)
   ├─ 2(s=2,d=1)
   │  ├─ 0(s=2,d=2)
   │  │  ├─ 0(s=2,d=3)
   │  │  ├─ 1(s=3,d=3)
   │  │  ├─ 2(s=4,d=3)
   │  │  ├─ 3(s=5,d=3)
   │  │  └─ 4(s=6,d=3)
   │  ├─ 1(s=3,d=2)
   │  │  ├─ 0(s=3,d=3)
   │  │  ├─ 1(s=4,d=3)
   │  │  ├─ 2(s=5,d=3)
   │  │  └─ 3(s=6,d=3)
   │  ├─ 2(s=4,d=2)
   │  │  ├─ 0(s=4,d=3)
   │  │  ├─ 1(s=5,d=3)
   │  │  └─ 2(s=6,d=3)
   │  ├─ 3(s=

# PAT-1105 链表合并
1. 先将所有节点读入进来，二维列表存储，直接储存为字符串格式。
2. 将L1和L2分离开来，对短的链表进行逆序。
3. 观察输出案例发现，短链表的左节点不变，右节点是长链表的左节点。意味着可以不对短链表进行真正的逆序。
4. 


In [ ]:
L1_head, L2_head, N = input().split()
N = int(N)
L = []
for _ in range(N):
    address, data, next_address = input().split()
    L.append([address, data, next_address])

def find_chain(head, L):
    chain = []
    address = head
    while address != -1:
        for node in L:
            if node[0] == address:
                chain.append(node)
                address = node[2]
                break
        else:
            break  # 如果没找到，跳出循环
    return chain

L1 = find_chain(L1_head, L)
L2 = find_chain(L2_head, L)
if len(L1) < len(L2):
    L1, L2 = L2, L1  # 确保L1是较长的链表

# L2逆序输出
L2.reverse()
# 合并链表
for i in range(len(L2)):

    # 更新插入点之前的节点的next地址
    L1[i*3+1][2] = L2[i][0]

    # 更新L2[i]的next地址
    L2[i][2] = L1[i*3+2][0]

    # 插入L2节点
    L1.insert(i*3+2, L2[i])

for i in L1:
    print(' '.join(i))

